# 03 · Model Training
Train 4 models: LogReg, Random Forest, XGBoost, LightGBM.


In [ ]:
from pathlib import Path
import json, joblib
import pandas as pd, numpy as np

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load feature cols and data (with backend fallback)
fpath = MODEL_DIR / "feature_columns.json"
if not fpath.exists():
    fpath = Path("../../apps/backend/ml_artifacts/feature_columns.json")
with open(fpath) as f:
    feature_cols = json.load(f)

spath = MODEL_DIR / "scaler.joblib"
if not spath.exists():
    spath = Path("../../apps/backend/ml_artifacts/scaler.joblib")
scaler = joblib.load(spath)

# Load X/y
if (DATA_DIR / "X_train.csv").exists():
    X_train = pd.read_csv(DATA_DIR / "X_train.csv")[feature_cols]
    y_train = pd.read_csv(DATA_DIR / "y_train.csv")["target_risk"].astype(int)
    X_test = pd.read_csv(DATA_DIR / "X_test.csv")[feature_cols]
    y_test = pd.read_csv(DATA_DIR / "y_test.csv")["target_risk"].astype(int)
else:
    train_m = pd.read_csv(DATA_DIR / "train_matrix.csv")
    test_m = pd.read_csv(DATA_DIR / "test_matrix.csv")
    X_train = train_m[feature_cols]
    y_train = train_m["target_risk"].astype(int)
    X_test = test_m[feature_cols]
    y_test = test_m["target_risk"].astype(int)

scale_w = float((y_train==0).sum() / max((y_train==1).sum(), 1))
print(f"X_train {X_train.shape} | X_test {X_test.shape} | scale_w {scale_w:.2f}")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, scale_pos_weight=scale_w, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1),
}

trained = {}
rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    trained[name] = m
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:,1]
    rows.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred),4),
        "Precision": round(precision_score(y_test, pred, zero_division=0),4),
        "Recall": round(recall_score(y_test, pred, zero_division=0),4),
        "F1": round(f1_score(y_test, pred, zero_division=0),4),
        "ROC-AUC": round(roc_auc_score(y_test, prob),4),
    })
    print(f"Trained {name}")

eval_df = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
display(eval_df)


In [ ]:
# Save models
for name, m in trained.items():
    slug = name.lower().replace(" ","_")
    joblib.dump(m, MODEL_DIR / f"{slug}.joblib")
    print(f"Saved {slug}.joblib")
eval_df.to_csv(MODEL_DIR / "model_evaluation.csv", index=False)
print(f"Saved evaluation -> {MODEL_DIR / 'model_evaluation.csv'}")
